# H3 — Graphes de connaissances et reponse a des questions

Ce notebook presente un mini-projet KGQA complet : une petite base RDF, une couche simple de liaison entite-texte, puis une traduction de questions en requetes SPARQL.

Objectif : montrer un pipeline clair, lisible et executable de bout en bout dans un notebook.

## Pipeline

Question en langage naturel -> detection des entites -> generation SPARQL -> execution sur le graphe -> reponse lisible.

Le choix ici est volontairement simple et robuste : un pipeline deterministe de demonstration, facile a expliquer en soutenance.

In [2]:
%pip install -q rdflib pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
from pathlib import Path

import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import XSD

pd.set_option("display.max_colwidth", None)

EX = Namespace("http://example.org/kgqa/")


def normalize(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")


g = Graph()
g.bind("ex", EX)
g.bind("rdfs", RDFS)

entity_aliases = {}


def register_entity(label: str, rdf_type):
    uri = EX[slug(label)]
    entity_aliases[normalize(label)] = uri
    g.add((uri, RDF.type, rdf_type))
    g.add((uri, RDFS.label, Literal(label)))
    return uri


Director = EX.Director
Actor = EX.Actor
Movie = EX.Movie

directors = {
    name: register_entity(name, Director) for name in ["Christopher Nolan", "James Cameron", "Alejandro G. Inarritu"]
}

actors = {
    name: register_entity(name, Actor)
    for name in [
        "Leonardo DiCaprio",
        "Matthew McConaughey",
        "Anne Hathaway",
        "Kate Winslet",
        "Sam Worthington",
        "Zoe Saldana",
        "Cillian Murphy",
    ]
}

movies = {
    "Inception": {
        "year": 2010,
        "genre": "Science Fiction",
        "director": directors["Christopher Nolan"],
        "cast": [actors["Leonardo DiCaprio"], actors["Cillian Murphy"]],
    },
    "Interstellar": {
        "year": 2014,
        "genre": "Science Fiction",
        "director": directors["Christopher Nolan"],
        "cast": [actors["Matthew McConaughey"], actors["Anne Hathaway"]],
    },
    "Titanic": {
        "year": 1997,
        "genre": "Romance",
        "director": directors["James Cameron"],
        "cast": [actors["Leonardo DiCaprio"], actors["Kate Winslet"]],
    },
    "Avatar": {
        "year": 2009,
        "genre": "Science Fiction",
        "director": directors["James Cameron"],
        "cast": [actors["Sam Worthington"], actors["Zoe Saldana"]],
    },
    "The Revenant": {
        "year": 2015,
        "genre": "Drama",
        "director": directors["Alejandro G. Inarritu"],
        "cast": [actors["Leonardo DiCaprio"]],
    },
    "Oppenheimer": {
        "year": 2023,
        "genre": "Drama",
        "director": directors["Christopher Nolan"],
        "cast": [actors["Cillian Murphy"]],
    },
}

for title, meta in movies.items():
    movie_uri = register_entity(title, Movie)
    g.add((movie_uri, EX.releaseYear, Literal(meta["year"], datatype=XSD.integer)))
    g.add((movie_uri, EX.genre, Literal(meta["genre"])))
    g.add((movie_uri, EX.directedBy, meta["director"]))
    for actor_uri in meta["cast"]:
        g.add((movie_uri, EX.starring, actor_uri))

print(f"Triples loaded: {len(g)}")
print(f"Entities registered: {len(entity_aliases)}")

Triples loaded: 60
Entities registered: 16


## Questions supportees

Le notebook couvre quelques formes de questions typiques :
- Qui a realise un film ?
- Quels films a realise une personne ?
- Qui joue dans un film ?
- Quels films feature un acteur ?
- Quel genre / quelle annee ?
- Quel realisateur a travaille sur des films avec un acteur donne ? (question multi-hop simple)

Cette version est volontairement contrainte pour etre stable en demo.

In [ ]:
PATTERNS = [
    ("director_of_movie", re.compile(r"^who directed (?P<entity>.+?)\??$", re.I)),
    ("movies_by_director", re.compile(r"^which movies did (?P<entity>.+?) direct\??$", re.I)),
    ("actors_in_movie", re.compile(r"^who starred in (?P<entity>.+?)\??$", re.I)),
    ("movies_with_actor", re.compile(r"^which movies feature (?P<entity>.+?)\??$", re.I)),
    ("genre_of_movie", re.compile(r"^what genre is (?P<entity>.+?)\??$", re.I)),
    ("movies_in_year", re.compile(r"^which movies were released in (?P<year>\d{4})\??$", re.I)),
    ("directors_of_actor", re.compile(r"^which directors worked on movies starring (?P<entity>.+?)\??$", re.I)),
]


def resolve_entity(text: str):
    key = normalize(text)
    if key in entity_aliases:
        return entity_aliases[key]
    matches = [uri for alias, uri in entity_aliases.items() if alias in key]
    if matches:
        return matches[0]
    raise KeyError(f"Unknown entity: {text}")


def label_query(uri_var: str, body: str) -> str:
    return f"""
PREFIX ex: <{EX}>
PREFIX rdfs: <{RDFS}>

SELECT DISTINCT ?answerLabel WHERE {{
  {body}
  {uri_var} rdfs:label ?answerLabel .
}}
"""


def question_to_sparql(question: str) -> str:
    q = question.strip()
    for intent, pattern in PATTERNS:
        match = pattern.match(q)
        if not match:
            continue
        data = match.groupdict()
        if intent == "director_of_movie":
            movie = resolve_entity(data["entity"])
            return label_query("?answer", f"<{movie}> ex:directedBy ?answer .")
        if intent == "movies_by_director":
            director = resolve_entity(data["entity"])
            return label_query("?answer", f"?answer ex:directedBy <{director}> .")
        if intent == "actors_in_movie":
            movie = resolve_entity(data["entity"])
            return label_query("?answer", f"<{movie}> ex:starring ?answer .")
        if intent == "movies_with_actor":
            actor = resolve_entity(data["entity"])
            return label_query("?answer", f"?answer ex:starring <{actor}> .")
        if intent == "genre_of_movie":
            movie = resolve_entity(data["entity"])
            return f"""
PREFIX ex: <{EX}>
PREFIX rdfs: <{RDFS}>

SELECT DISTINCT ?answerLabel WHERE {{
  <{movie}> ex:genre ?answerLabel .
}}
"""
        if intent == "movies_in_year":
            year = data["year"]
            return label_query("?answer", f"?answer ex:releaseYear {year} .")
        if intent == "directors_of_actor":
            actor = resolve_entity(data["entity"])
            return label_query("?answer", f"?movie ex:starring <{actor}> ; ex:directedBy ?answer .")
    raise ValueError(f"Unsupported question: {question}")


def answer_question(question: str) -> tuple[str, pd.DataFrame]:
    query = question_to_sparql(question)
    rows = [str(row[0]) for row in g.query(query)]
    return query, pd.DataFrame({"answer": rows})


print("Question translator ready.")

Question translator ready.


In [5]:
examples = [
    "Who directed Inception?",
    "Which movies did Christopher Nolan direct?",
    "Who starred in Titanic?",
    "Which movies feature Leonardo DiCaprio?",
    "What genre is Interstellar?",
    "Which movies were released in 2010?",
    "Which directors worked on movies starring Leonardo DiCaprio?",
]

for question in examples:
    query, answers = answer_question(question)
    print(f"\nQ: {question}")
    print(query.strip())
    if answers.empty:
        print("A: No result")
    else:
        print("A:")
        print(answers.to_string(index=False))


Q: Who directed Inception?
PREFIX ex: <http://example.org/kgqa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?answerLabel WHERE {
  <http://example.org/kgqa/Inception> ex:directedBy ?answer .
  ?answer rdfs:label ?answerLabel .
}
A:
           answer
Christopher Nolan

Q: Which movies did Christopher Nolan direct?
PREFIX ex: <http://example.org/kgqa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?answerLabel WHERE {
  ?answer ex:directedBy <http://example.org/kgqa/Christopher_Nolan> .
  ?answer rdfs:label ?answerLabel .
}
A:
      answer
   Inception
Interstellar
 Oppenheimer

Q: Who starred in Titanic?
PREFIX ex: <http://example.org/kgqa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?answerLabel WHERE {
  <http://example.org/kgqa/Titanic> ex:starring ?answer .
  ?answer rdfs:label ?answerLabel .
}
A:
           answer
Leonardo DiCaprio
     Kate Winslet

Q: Which movies feature Leonardo DiCaprio?
PREFIX ex: 

In [6]:
your_question = "Which movies did James Cameron direct?"
query, answers = answer_question(your_question)
print(f"Q: {your_question}")
print(query.strip())
print(answers.to_string(index=False) if not answers.empty else "No result")

Q: Which movies did James Cameron direct?
PREFIX ex: <http://example.org/kgqa/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?answerLabel WHERE {
  ?answer ex:directedBy <http://example.org/kgqa/James_Cameron> .
  ?answer rdfs:label ?answerLabel .
}
 answer
Titanic
 Avatar


## Limites et pistes d'amelioration

Ce prototype est volontairement template-based : il fonctionne tres bien pour des formulations prevues, mais il generalise mal aux paraphrases.

Pour aller plus loin, on pourrait ajouter :
- un vrai module d'entity linking,
- un parseur semantique plus general,
- une version LLM-assisted pour produire le SPARQL,
- un benchmark avec plusieurs formulations par question.

## Mini-evaluation

Le README demande une evaluation ; ici on ajoute un test reproductible sur un petit jeu de questions de validation.

Ce n'est pas le benchmark complet LC-QuAD/WebQuestionsSP, mais cela permet de mesurer la precision du pipeline de demonstration sur des cas connus.

In [ ]:
evaluation_set = [
    ("Who directed Inception?", {"Christopher Nolan"}),
    ("Which movies did Christopher Nolan direct?", {"Inception", "Interstellar", "Oppenheimer"}),
    ("Who starred in Titanic?", {"Leonardo DiCaprio", "Kate Winslet"}),
    ("Which movies feature Leonardo DiCaprio?", {"Inception", "Titanic", "The Revenant"}),
    ("What genre is Interstellar?", {"Science Fiction"}),
]

rows = []
for question, expected in evaluation_set:
    _, df = answer_question(question)
    predicted = set(df["answer"].tolist())
    rows.append(
        {
            "question": question,
            "expected": ", ".join(sorted(expected)),
            "predicted": ", ".join(sorted(predicted)),
            "exact_match": predicted == expected,
        }
    )

eval_df = pd.DataFrame(rows)
accuracy = eval_df["exact_match"].mean()
eval_df, accuracy

(                                     question  \
 0                     Who directed Inception?   
 1  Which movies did Christopher Nolan direct?   
 2                     Who starred in Titanic?   
 3     Which movies feature Leonardo DiCaprio?   
 4                 What genre is Interstellar?   
 
                                expected                             predicted  \
 0                     Christopher Nolan                     Christopher Nolan   
 1  Inception, Interstellar, Oppenheimer  Inception, Interstellar, Oppenheimer   
 2       Kate Winslet, Leonardo DiCaprio       Kate Winslet, Leonardo DiCaprio   
 3      Inception, The Revenant, Titanic      Inception, The Revenant, Titanic   
 4                       Science Fiction                       Science Fiction   
 
    exact_match  
 0         True  
 1         True  
 2         True  
 3         True  
 4         True  ,
 np.float64(1.0))

## Conclusion

Le notebook fournit une demonstration claire du sujet H3 : passer d'une question en langage naturel a une requete SPARQL executable sur un graphe RDF.

Il est concis, demonstratif et simple a presenter en cours ou en soutenance.